In [28]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
import re

In [29]:
df = pd.read_csv("../data/parquets/processed_parquet.csv")

In [30]:
def val_count(data):
    counter = 0
    for i in data.columns:        
        print(counter)
        print(data[i].value_counts())
        counter += 1
        print("=========================="*5)
    print(data.shape)

In [31]:
# for i in df.columns:
#     if "volcanic" in i:
#         print(f"'{i}',")
        

In [36]:
def find_purpose(spot, purpose_features, purpose_threshold = False):
    THRESHOLD = 1.5
    features_sum = 0
    not_features_sum = 0
        
        
    if not(purpose_threshold):
        for feature in spot.index:
            if spot[feature] == 1 and feature in purpose_features:
                return True
        return False
    
    else:
        
        for feature in spot.index:
            if spot[feature] == 1 and feature in purpose_features:
                features_sum += 1
            elif spot[feature] == 1 and feature not in purpose_features:
                not_features_sum += 1
                
        purpose_presence = features_sum
        not_purpose_presence = not_features_sum
        
        # if no features that are not a part of the purpose are present, assume spot is the purpose
        if not_purpose_presence == 0:
            return True
        
        if purpose_presence / not_purpose_presence >= THRESHOLD: 
            return True
        else:
            return False

In [33]:
gm_features = ["properties_rock_unit_unit_label_abbreviation",
               "properties_images_notes",
               "properties_images",
               "properties_notes",
               "properties_pet_rock_type",
               "geometry_coordinates",
               "properties_images"
               "properties_orientation_data", 
               "properties_trace_trace_feature", 
               "properties_trace_trace_quality", 
               "properties_trace_trace_type", 
               "properties_trace_contact_type", 
               "properties_other_features"
               ]

site_safety_features = ["properties_site_safety_site_summary_author",
                        "properties_site_safety_field_stop_designation",
                        "properties_site_safety_longitude",
                        "properties_site_safety_latitude"
                        ]

strat_features = [ "properties_sed_strat_section_column_profile",
                  "properties_sed_strat_section_column_y_axis_units",
                  "properties_sed_strat_section_purpose",
                  "properties_sed_strat_section_scale_of_interest",
                  "properties_sed_strat_section_obs_interval_bed_obs_scale",
                  "properties_sed_strat_section_section_type",
                  "properties_sed_strat_section_basin",
                  "properties_sed_strat_section_age",
                  "properties_sed_strat_section_total_core_length",
                  "properties_sed_strat_section_images",
                  "properties_sed_strat_section_section_well_name",
                  "properties_sed_strat_section_location_locality",
                  "properties_sed_strat_section_dates_of_work",
                  "properties_sed_strat_section_strat_section_notes",
                  "properties_sed_strat_section_display_lithology_patterns",
                  "properties_sed_strat_section_project_description",
                  "properties_sed_strat_section_how_is_section_georeferenced",
                  "properties_sed_strat_section_misc_labels",
                  "properties_sed_strat_section_what_core_repository",
                  "properties_sed_strat_section_type_of_corer",
                  "properties_sed_strat_section_depth_from_surface_to_start_of",
                  "properties_sed_strat_section_other_purpose",
                  "properties_sed_strat_section_label",
                  "properties_sed_strat_section_other_scale_of_interest"
                  ]

petrology_features = [
    'properties_pet_rock_type',
    'properties_pet_igneous_rock_class',
    'properties_pet_volcanic_rock_type',
    'properties_pet_occurence_volcanic',
    'properties_pet_metamorphic_rock_type',
    'properties_pet_protolith',
    'properties_pet_texture_volcanic',
    'properties_pet_alteration_volcanic',
    'properties_pet_plutonic_rock_type',
    'properties_pet_occurence_plutonic',
    'properties_pet_texture_plutonic',
    'properties_pet_minerals',
    'properties_pet_other_protolith',
    'properties_pet_notes_metamorphic',
    'properties_pet_facies',
    'properties_pet_zone',
    'properties_pet_color_index_source_pluton',
    'properties_pet_color_index_pluton',
    'properties_pet_pluton_characteristic_size_of',
    'properties_pet_alteration_plutonic',
    'properties_pet_notes_plutonic',
    'properties_pet_other_texture_plutonic',
    'properties_pet_igneous',
    'properties_pet_metamorphic',
    'properties_pet_metamorphic_rock_type_less_com',
    'properties_pet_reactions',
    'properties_pet_color_index_volc',
    'properties_pet_vol_characteristic_size_of_cry',
    'properties_pet_other_plutonic',
    'properties_pet_other_met_rocks',
    'properties_pet_other_volcanic_rocks',
    'properties_pet_fault',
    'properties_pet_other_occurence_plutonic',
    'properties_pet_notes_volcanic',
    'properties_pet_structure_volcanic',
    'properties_pet_color_index_source_volc',
    'properties_pet_other_modification_plutonic',
    'properties_pet_alteration_or',
    'properties_pet_other_occurance_volcanic'
]

struct_geo_features = [
    "properties_orientation_type",
    "properties_orientation_strike",
    "properties_orientation_dip_direction",
    "properties_orientation_dip",
    "properties_orientation_trend",
    "properties_sed_bedding_package_bedding_trends",
    "properties_sed_lithologies_bedding_grain_size_trends",
    "properties_sed_lithologies_package_grain_size_trends",
    "properties_orientation_plunge",
    "properties_trace_geologic_structure_type",
    "properties_orientation_foliation_type"
]

volcanology_features = [
    "properties_tephra",
    "properties_pet_igneous",
    'properties_pet_volcanic_rock_type',
    'properties_pet_occurence_volcanic',
    'properties_pet_texture_volcanic',
    'properties_pet_alteration_volcanic',
    'properties_pet_other_volcanic_rocks',
    'properties_sed_lithologies_volcaniclastic_type',
    'properties_sed_lithologies_other_volcaniclastic_type',
    'properties_pet_notes_volcanic',
    'properties_pet_structure_volcanic',
    'properties_pet_other_occurance_volcanic',
]




In [34]:
tectonic_features = [
    "properties_rock_unit_period",
    "properties_sed_interpretations_tectonic_setting",
]


sedimentology_features = [
    "properties_sed_bedding_beds",
    "properties_sed_bedding_lithology_at_bottom_contact",
    "properties_sed_bedding_lithology_at_top_contact"
]


In [37]:
df["is_general_mapping"] = df.apply(lambda x: find_purpose(x,gm_features), axis = 1)
df["is_site_safety"] = df.apply(lambda x: find_purpose(x,site_safety_features), axis = 1)
df["is_strat"] = df.apply(lambda x: find_purpose(x,strat_features), axis = 1)
df["is_pet"] = df.apply(lambda x: find_purpose(x,petrology_features), axis = 1)
df["is_struct_geo"] = df.apply(lambda x: find_purpose(x,struct_geo_features), axis = 1)
df["is_volcanology"] = df.apply(lambda x: find_purpose(x,volcanology_features), axis = 1)

In [38]:
val_count(df.iloc[:, 440:])

0
is_general_mapping
True     1220926
False       8448
Name: count, dtype: int64
1
is_site_safety
False    1229363
True          11
Name: count, dtype: int64
2
is_strat
False    1228207
True        1167
Name: count, dtype: int64
3
is_pet
False    1223057
True        6317
Name: count, dtype: int64
4
is_struct_geo
False    1208910
True       20464
Name: count, dtype: int64
5
is_volcanology
False    1227347
True        2027
Name: count, dtype: int64
(1229374, 6)


In [39]:
df["is_general_mapping"] = df.apply(lambda x: find_purpose(x,gm_features, purpose_threshold= True), axis = 1)
df["is_site_safety"] = df.apply(lambda x: find_purpose(x,site_safety_features, purpose_threshold= True), axis = 1)
df["is_strat"] = df.apply(lambda x: find_purpose(x,strat_features, purpose_threshold= True), axis = 1)
df["is_pet"] = df.apply(lambda x: find_purpose(x,petrology_features, purpose_threshold= True), axis = 1)
df["is_struct_geo"] = df.apply(lambda x: find_purpose(x,struct_geo_features, purpose_threshold= True), axis = 1)
df["is_volcanology"] = df.apply(lambda x: find_purpose(x,volcanology_features, purpose_threshold= True), axis = 1)

In [40]:
val_count(df.iloc[:, 440:])

0
is_general_mapping
False    1229374
Name: count, dtype: int64
1
is_site_safety
False    1229374
Name: count, dtype: int64
2
is_strat
False    1229022
True         352
Name: count, dtype: int64
3
is_pet
False    1229297
True          77
Name: count, dtype: int64
4
is_struct_geo
False    1229374
Name: count, dtype: int64
5
is_volcanology
False    1229374
Name: count, dtype: int64
(1229374, 6)


In [ ]:
df['features_present'] = df.iloc[:, :440].sum(axis=1, numeric_only=True)

In [52]:
float(df["features_present"].mean())

4.412614875538282

In [56]:
# print(df["features_present"].to_string())